# Cam-Follower Timing Diagram

Cam-follower mechanisms convert rotary motion into precise translating or
oscillating motion. This notebook demonstrates pylinkage's cam module:

- Defining cam profiles with different motion laws
- Comparing displacement, velocity, and acceleration curves
- Building a translating cam-follower mechanism
- Building an oscillating cam-follower mechanism
- Analyzing pressure angles for design safety

**Use case:** A cam-driven valve mechanism with rise-dwell-fall-dwell timing.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from pylinkage.cam import (
    CycloidalMotionLaw,
    FunctionProfile,
    HarmonicMotionLaw,
    ModifiedTrapezoidalMotionLaw,
    PointArrayProfile,
    polynomial_345,
    polynomial_4567,
)

## 1. Define a Cam Profile

A valve cam with the following timing:
- **Rise** from 0 to 90 degrees (valve opens)
- **Dwell high** from 90 to 180 degrees (valve fully open)
- **Fall** from 180 to 270 degrees (valve closes)
- **Dwell low** from 270 to 360 degrees (valve closed)

In [ ]:
base_radius = 2.0    # Base circle radius (cm)
total_lift = 1.0     # Follower lift (cm)

# Timing angles (radians)
rise_start = 0.0
rise_end = math.pi / 2          # 90 deg
dwell_high_end = math.pi        # 180 deg
fall_end = 3 * math.pi / 2      # 270 deg
# Dwell low continues to 360 deg (2*pi)

# Create a profile with harmonic motion law
harmonic_profile = FunctionProfile(
    motion_law=HarmonicMotionLaw(),
    base_radius=base_radius,
    total_lift=total_lift,
    rise_start=rise_start,
    rise_end=rise_end,
    dwell_high_end=dwell_high_end,
    fall_end=fall_end,
    name="harmonic_valve",
)

print(f"At 0 deg: r = {harmonic_profile.evaluate(0):.3f} cm")
print(f"At 90 deg: r = {harmonic_profile.evaluate(math.pi/2):.3f} cm")
print(f"At 180 deg: r = {harmonic_profile.evaluate(math.pi):.3f} cm")

## 2. Compare Motion Laws

Different motion laws trade off smoothness, peak velocity, and peak acceleration.
Let's compare the common options.

In [ ]:
motion_laws = {
    'Harmonic': HarmonicMotionLaw(),
    'Cycloidal': CycloidalMotionLaw(),
    'Mod. Trapezoidal': ModifiedTrapezoidalMotionLaw(),
    'Polynomial 3-4-5': polynomial_345(),
    'Polynomial 4-5-6-7': polynomial_4567(),
}

profiles = {}
for name, law in motion_laws.items():
    profiles[name] = FunctionProfile(
        motion_law=law,
        base_radius=base_radius,
        total_lift=total_lift,
        rise_start=rise_start,
        rise_end=rise_end,
        dwell_high_end=dwell_high_end,
        fall_end=fall_end,
    )

In [ ]:
angles_deg = np.linspace(0, 360, 500)
angles_rad = np.radians(angles_deg)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Displacement diagram
ax1 = axes[0]
for name, profile in profiles.items():
    displacement = [profile.evaluate(a) - base_radius for a in angles_rad]
    ax1.plot(angles_deg, displacement, linewidth=1.5, label=name)

ax1.set_ylabel('Follower displacement (cm)')
ax1.set_title('Cam Timing Diagram — Displacement')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Add timing zones
for ax in axes:
    ax.axvspan(0, 90, alpha=0.05, color='green', label='_')
    ax.axvspan(90, 180, alpha=0.05, color='blue')
    ax.axvspan(180, 270, alpha=0.05, color='red')
    ax.axvspan(270, 360, alpha=0.05, color='gray')

# Velocity diagram (derivative)
ax2 = axes[1]
for name, profile in profiles.items():
    velocity = [profile.evaluate_derivative(a) for a in angles_rad]
    ax2.plot(angles_deg, velocity, linewidth=1.5, label=name)

ax2.set_xlabel('Cam angle (degrees)')
ax2.set_ylabel('dr/d\u03b8 (cm/rad)')
ax2.set_title('Cam Timing Diagram — Velocity (dr/d\u03b8)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# Annotate timing zones
for x, label in [(45, 'RISE'), (135, 'DWELL\nHIGH'), (225, 'FALL'), (315, 'DWELL\nLOW')]:
    axes[0].text(x, total_lift * 1.05, label, ha='center', va='bottom',
                 fontsize=8, color='gray')

plt.tight_layout()
plt.show()

## 3. Cam Profile Polar Plot

The actual cam shape is best visualized as a polar plot.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})

for name, profile in profiles.items():
    radii = [profile.evaluate(a) for a in angles_rad]
    ax.plot(angles_rad, radii, linewidth=1.5, label=name)

# Base circle
ax.plot(angles_rad, [base_radius] * len(angles_rad), 'k--', linewidth=0.5,
        alpha=0.5, label='Base circle')

ax.set_title('Cam profiles (polar view)', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=9)
plt.tight_layout()
plt.show()

## 4. Pressure Angle Analysis

Pressure angle measures the force transmission efficiency. A pressure angle above
30 degrees can cause binding in translating followers. This is a key design check.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for name, profile in profiles.items():
    pressure = [math.degrees(profile.pressure_angle(a)) for a in angles_rad]
    ax.plot(angles_deg, pressure, linewidth=1.5, label=name)

# Safe limits
ax.axhline(y=30, color='red', linestyle='--', alpha=0.5, label='Max safe (30 deg)')
ax.axhline(y=-30, color='red', linestyle='--', alpha=0.5)

ax.set_xlabel('Cam angle (degrees)')
ax.set_ylabel('Pressure angle (degrees)')
ax.set_title('Pressure Angle Comparison')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Report max pressure angles
print("Peak pressure angles:")
for name, profile in profiles.items():
    max_pa = max(abs(math.degrees(profile.pressure_angle(a))) for a in angles_rad)
    status = 'OK' if max_pa <= 30 else 'EXCEEDS LIMIT'
    print(f"  {name:25s}: {max_pa:.1f} deg  [{status}]")

## 5. Build a Translating Cam-Follower Mechanism

Now let's build a complete mechanism with a cam driving a translating follower.

In [ ]:
from pylinkage.actuators import Crank
from pylinkage.components import Ground
from pylinkage.dyads import TranslatingCamFollower
from pylinkage.simulation import Linkage

# Cam center
cam_origin = Ground(0.0, 0.0, name="cam_center")

# Cam driver (angular velocity = angle step per iteration)
cam_driver = Crank(
    anchor=cam_origin,
    radius=0.1,  # Small radius (cam rotation reference)
    angular_velocity=2 * math.pi / 200,  # Full rotation in 200 steps
    name="cam",
)

# Follower guide origin
guide = Ground(0.0, 0.0, name="guide")

# Select the cycloidal profile (smooth motion)
cam_profile = profiles['Cycloidal']

# Create translating follower moving vertically
follower = TranslatingCamFollower(
    cam_driver=cam_driver,
    profile=cam_profile,
    guide=guide,
    guide_angle=math.pi / 2,  # Vertical motion
    roller_radius=0.2,
    name="follower",
)

# Build linkage
mechanism = Linkage(
    components=[cam_origin, cam_driver, guide, follower],
    name="cam_valve",
)

print(f"Components: {[c.name for c in mechanism.components]}")

In [ ]:
# Simulate
loci = list(mechanism.step(iterations=200))

# Extract follower position over time
follower_idx = -1  # Last component
follower_y = [pos[follower_idx][1] for pos in loci if pos[follower_idx][1] is not None]

fig, ax = plt.subplots(figsize=(10, 4))
time_deg = np.linspace(0, 360, len(follower_y))
ax.plot(time_deg, follower_y, 'b-', linewidth=2)
ax.set_xlabel('Cam angle (degrees)')
ax.set_ylabel('Follower position Y (cm)')
ax.set_title('Translating cam-follower: simulated motion')
ax.grid(True)
plt.tight_layout()
plt.show()

## 6. Custom Profile from Discrete Points

For non-standard motions, define a cam profile from measured or desired points.
Spline interpolation fills in the gaps.

In [ ]:
# Define custom profile from discrete data
custom_angles = [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360]
custom_radii =  [2.0, 2.1, 2.4, 2.8, 3.0, 3.0, 3.0, 2.8, 2.4, 2.0, 2.0, 2.0, 2.0]

custom_profile = PointArrayProfile(
    angles=[math.radians(a) for a in custom_angles],
    radii=custom_radii,
    name="custom_valve",
)

# Compare with analytical profile
fig, ax = plt.subplots(figsize=(10, 5))

# Custom profile (interpolated)
custom_r = [custom_profile.evaluate(a) for a in angles_rad]
ax.plot(angles_deg, custom_r, 'r-', linewidth=2, label='Custom (spline)')
ax.plot(custom_angles, custom_radii, 'ro', markersize=8, label='Input points')

# Cycloidal for comparison
cycloidal_r = [profiles['Cycloidal'].evaluate(a) for a in angles_rad]
ax.plot(angles_deg, cycloidal_r, 'b--', linewidth=1.5, label='Cycloidal')

ax.set_xlabel('Cam angle (degrees)')
ax.set_ylabel('Cam radius (cm)')
ax.set_title('Custom vs analytical cam profile')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Motion Law | Pros | Cons | Best For |
|---|---|---|---|
| Harmonic | Simple, smooth | Non-zero velocity at dwell transitions | Low-speed cams |
| Cycloidal | Zero velocity & accel at boundaries | Higher peak velocity | High-speed cams |
| Mod. Trapezoidal | Lowest peak acceleration | More complex | Heavy followers |
| Polynomial 3-4-5 | Zero velocity at boundaries | Moderate peak accel | General purpose |
| Polynomial 4-5-6-7 | Zero velocity & accel at boundaries | Highest peak velocity | Precision cams |

**Key design checks:**
- Pressure angle < 30 degrees for translating followers
- Pressure angle < 45 degrees for oscillating followers
- Use `PointArrayProfile` for irregular or measured profiles